In [1]:
# Cell 1: Load the ATO-labeled dataframe from previous notebook
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/processed/df_ato_labeled.parquet')
print(f"Loaded: {df.shape}")
print(f"ato_proxy=1 : {df['ato_proxy'].sum():,}")
print(f"ato_proxy=0 : {(df['ato_proxy']==0).sum():,}")
print(f"ATO rate    : {df['ato_proxy'].mean()*100:.3f}%")

Loaded: (590540, 446)
ato_proxy=1 : 2,362
ato_proxy=0 : 588,178
ATO rate    : 0.400%


In [2]:
# Cell 2: Select features most relevant to ATO detection
# Dropping Vesta's masked V1-V339 columns — too many, too sparse for sequence modelling
# Focusing on identity, device, temporal and card-level features

identity_cols = [f'id_0{i}' for i in range(1, 10)] + \
                ['id_10', 'id_11'] + \
                [f'id_{i}' for i in range(12, 39)]
identity_cols = [c for c in identity_cols if c in df.columns]

device_cols = ['DeviceType', 'DeviceInfo']

temporal_cols = ['TransactionDT', 'TransactionAmt']

card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']
card_cols = [c for c in card_cols if c in df.columns]

addr_cols = ['addr1', 'addr2']
addr_cols = [c for c in addr_cols if c in df.columns]

signal_cols = ['signal_device', 'signal_session', 'signal_velocity',
               'signal_count', 'device_changed', 'time_since_prev']

keep_cols = (
    ['TransactionID', 'uid', 'ato_proxy', 'isFraud'] +
    identity_cols + device_cols + temporal_cols +
    card_cols + addr_cols + signal_cols
)

keep_cols = [c for c in keep_cols if c in df.columns]
df_feat = df[keep_cols].copy()

print(f"Selected {len(keep_cols)} columns")
print(f"Identity cols  : {len(identity_cols)}")
print(f"Signal cols    : {len(signal_cols)}")
print(f"Card/addr cols : {len(card_cols) + len(addr_cols)}")

Selected 60 columns
Identity cols  : 38
Signal cols    : 6
Card/addr cols : 8


In [3]:
# Cell 3: Encode categorical columns — LSTM requires all-numeric input
# DeviceType and card categoricals converted to integer codes

# DeviceType: mobile=1, desktop=2, unknown=0
df_feat['DeviceType_enc'] = df_feat['DeviceType'].map(
    {'mobile': 1, 'desktop': 2}
).fillna(0).astype(int)

# DeviceInfo: too many unique values — encode as integer rank by frequency
device_freq = df_feat['DeviceInfo'].value_counts()
df_feat['DeviceInfo_enc'] = df_feat['DeviceInfo'].map(device_freq).fillna(0).astype(int)

# Card categoricals
for col in ['card4', 'card6']:
    if col in df_feat.columns:
        df_feat[f'{col}_enc'] = df_feat[col].astype('category').cat.codes

# Drop original string columns
df_feat = df_feat.drop(columns=['DeviceType', 'DeviceInfo'], errors='ignore')
df_feat = df_feat.drop(columns=['card4', 'card6'], errors='ignore')

print("Categorical encoding complete")
print(f"DeviceType_enc distribution:\n{df_feat['DeviceType_enc'].value_counts()}")

Categorical encoding complete
DeviceType_enc distribution:
0    449730
2     85165
1     55645
Name: DeviceType_enc, dtype: int64


In [4]:
# Cell 4: Handle missing values
# Strategy: median imputation for numeric columns
# Rationale: identity columns are missing for 76% of transactions (EDA finding)
# Mean would be skewed by outliers; median is more robust

numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
exclude = ['TransactionID', 'ato_proxy', 'isFraud']
impute_cols = [c for c in numeric_cols if c not in exclude]

# Store medians for later use in inference
medians = df_feat[impute_cols].median()
df_feat[impute_cols] = df_feat[impute_cols].fillna(medians)

print(f"Imputed {len(impute_cols)} columns with median values")
print(f"Remaining nulls: {df_feat[impute_cols].isnull().sum().sum()}")

# Save medians for inference pipeline
medians.to_csv('../outputs/imputation_medians.csv')
print("Saved: outputs/imputation_medians.csv")

Imputed 41 columns with median values
Remaining nulls: 0
Saved: outputs/imputation_medians.csv


In [5]:
# Cell 5: Engineer per-card temporal and behavioural features
# These features capture patterns across a card's transaction history

df_feat = df_feat.sort_values(['uid', 'TransactionDT']).reset_index(drop=True)

# Rolling transaction count per card (last 10 transactions)
df_feat['txn_count_rolling'] = df_feat.groupby('uid').cumcount() + 1

# Amount deviation from card's historical mean
card_amt_mean = df_feat.groupby('uid')['TransactionAmt'].transform('mean')
card_amt_std  = df_feat.groupby('uid')['TransactionAmt'].transform('std').fillna(1)
df_feat['amt_deviation'] = (df_feat['TransactionAmt'] - card_amt_mean) / card_amt_std

# Time since first transaction on this card (account age proxy)
card_first_dt = df_feat.groupby('uid')['TransactionDT'].transform('min')
df_feat['account_age_dt'] = df_feat['TransactionDT'] - card_first_dt

# Hour of day proxy (TransactionDT mod 86400 / 3600)
df_feat['hour_of_day'] = (df_feat['TransactionDT'] % 86400) / 3600

print("=== Engineered Features ===")
print(f"amt_deviation  — mean: {df_feat['amt_deviation'].mean():.4f}, std: {df_feat['amt_deviation'].std():.4f}")
print(f"account_age_dt — max: {df_feat['account_age_dt'].max():,} seconds")
print(f"hour_of_day    — range: {df_feat['hour_of_day'].min():.1f} – {df_feat['hour_of_day'].max():.1f}")

=== Engineered Features ===
amt_deviation  — mean: nan, std: nan
account_age_dt — max: 15,723,425 seconds
hour_of_day    — range: 0.0 – 24.0


In [6]:
# Cell 5b: Define feature_cols — must run before Cell 6a and Cell 6b
feature_cols = [
    'TransactionAmt', 'amt_deviation', 'account_age_dt',
    'hour_of_day', 'txn_count_rolling', 'time_since_prev',
    'signal_count', 'DeviceInfo_enc', 'DeviceType_enc'
] + [c for c in df_feat.columns if c.startswith('id_0') or
                                    c.startswith('id_1') or
                                    c.startswith('id_2') or
                                    c.startswith('id_3')]

# Keep only numeric columns that exist in df_feat
feature_cols = [
    c for c in feature_cols
    if c in df_feat.columns and pd.api.types.is_numeric_dtype(df_feat[c])
]

# Convert any remaining object dtype to numeric
for col in feature_cols:
    df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce').fillna(0)

print(f"feature_cols defined: {len(feature_cols)} columns")

feature_cols defined: 32 columns


In [7]:
# Cell 6a: Clean infinities and extreme values before scaling
import numpy as np

# Replace inf and -inf with NaN, then fill with 0
df_feat[feature_cols] = df_feat[feature_cols].replace([np.inf, -np.inf], np.nan)

# Check how many infinities were found
inf_count = df_feat[feature_cols].isnull().sum().sum()
print(f"Infinity/NaN values replaced: {inf_count:,}")

# Fill remaining NaN with column median
for col in feature_cols:
    median_val = df_feat[col].median()
    df_feat[col] = df_feat[col].fillna(median_val)

# Clip extreme outliers (beyond 99.9th percentile) to prevent scaling issues
for col in feature_cols:
    upper = df_feat[col].quantile(0.999)
    lower = df_feat[col].quantile(0.001)
    df_feat[col] = df_feat[col].clip(lower=lower, upper=upper)

print("Infinities cleaned, outliers clipped")
print(f"Any remaining NaN: {df_feat[feature_cols].isnull().sum().sum()}")
print(f"Any remaining inf: {np.isinf(df_feat[feature_cols].values).sum()}")

Infinity/NaN values replaced: 101
Infinities cleaned, outliers clipped
Any remaining NaN: 0
Any remaining inf: 0


In [8]:
# Cell 6b: Scale features
from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()
df_feat[feature_cols] = scaler.fit_transform(df_feat[feature_cols])

with open('../outputs/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"Scaled {len(feature_cols)} features")
print("Saved: outputs/scaler.pkl")
print(f"\nSample after scaling (TransactionAmt):")
print(df_feat['TransactionAmt'].describe())

Scaled 32 features
Saved: outputs/scaler.pkl

Sample after scaling (TransactionAmt):
count    5.905400e+05
mean     4.562567e-17
std      1.000001e+00
min     -5.840000e-01
25%     -4.066385e-01
50%     -2.929750e-01
75%     -4.181930e-02
max      1.177121e+01
Name: TransactionAmt, dtype: float64


In [9]:
# Cell 7: Build sliding window sequences for LSTM
# Each sequence = last 10 transactions for a given card
# Label = ato_proxy value of the final transaction in the sequence

WINDOW_SIZE = 10

sequences = []
labels    = []

for uid, group in df_feat.groupby('uid'):
    group = group.sort_values('TransactionDT').reset_index(drop=True)
    feat_vals = group[feature_cols].values
    label_vals = group['ato_proxy'].values

    if len(group) < 2:
        continue  # skip cards with only 1 transaction

    for i in range(1, len(group)):
        start = max(0, i - WINDOW_SIZE)
        seq   = feat_vals[start:i]

        # Pad shorter sequences with zeros at the front
        if len(seq) < WINDOW_SIZE:
            pad = np.zeros((WINDOW_SIZE - len(seq), len(feature_cols)))
            seq = np.vstack([pad, seq])

        sequences.append(seq)
        labels.append(label_vals[i])

X = np.array(sequences)
y = np.array(labels)

print(f"X shape : {X.shape}  →  (samples, timesteps, features)")
print(f"y shape : {y.shape}")
print(f"Positive class (ato_proxy=1): {y.sum():,} ({y.mean()*100:.3f}%)")

X shape : (550566, 10, 32)  →  (samples, timesteps, features)
y shape : (550566,)
Positive class (ato_proxy=1): 2,362 (0.429%)


# Cell 8: Apply SMOTE to address class imbalance, then save
from imblearn.over_sampling import SMOTE

# Reshape X for SMOTE: (samples, timesteps * features)
n_samples, n_steps, n_feats = X.shape
X_flat = X.reshape(n_samples, n_steps * n_feats)

smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_flat_res, y_res = smote.fit_resample(X_flat, y)

# Reshape back to 3D
X_res = X_flat_res.reshape(-1, n_steps, n_feats)

print(f"=== After SMOTE ===")
print(f"X shape     : {X_res.shape}")
print(f"y shape     : {y_res.shape}")
print(f"ATO=1 count : {y_res.sum():,} ({y_res.mean()*100:.2f}%)")

# Save sequences
np.save('../data/processed/X_sequences.npy', X_res)
np.save('../data/processed/y_labels.npy', y_res)

print("\nSaved: data/processed/X_sequences.npy")
print("Saved: data/processed/y_labels.npy")
print(f"\nNext step -> 04_lstm_model.ipynb")

In [11]:
# Cell 8: Save sequences without SMOTE
# Class imbalance handled via class_weight='balanced' in LSTM training
# This avoids distribution shift between SMOTE-augmented validation and real test data
np.save('../data/processed/X_sequences.npy', X)
np.save('../data/processed/y_labels.npy', y)

print("Saved: data/processed/X_sequences.npy")
print("Saved: data/processed/y_labels.npy")
print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"ATO=1   : {y.sum():,} ({y.mean()*100:.3f}%)")
print(f"\nNote: Class imbalance handled via class_weight='balanced' in LSTM training")
print(f"\nNext step -> 04_lstm_model.ipynb")

Saved: data/processed/X_sequences.npy
Saved: data/processed/y_labels.npy

X shape : (550566, 10, 32)
y shape : (550566,)
ATO=1   : 2,362 (0.429%)

Note: Class imbalance handled via class_weight='balanced' in LSTM training

Next step -> 04_lstm_model.ipynb
